# Liquid Neural Network market index prediction

Simulate synthetic tick-by-tick market data.


In [1]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

print("Libraries numpy, pandas, and datetime imported successfully.")

Libraries numpy, pandas, and datetime imported successfully.


In [2]:
start_time = datetime(2023, 1, 1, 9, 0, 0) # January 1, 2023, 9:00:00 AM
num_ticks = 1000 # Number of data points
tick_frequency_seconds = 1 # Tick every 1 second

# Generate a series of timestamps
timestamps = [start_time + timedelta(seconds=i * tick_frequency_seconds) for i in range(num_ticks)]

print(f"Generated {num_ticks} timestamps starting from {start_time} with {tick_frequency_seconds}-second frequency.")
print(f"First 5 timestamps: {timestamps[:5]}")

Generated 1000 timestamps starting from 2023-01-01 09:00:00 with 1-second frequency.
First 5 timestamps: [datetime.datetime(2023, 1, 1, 9, 0), datetime.datetime(2023, 1, 1, 9, 0, 1), datetime.datetime(2023, 1, 1, 9, 0, 2), datetime.datetime(2023, 1, 1, 9, 0, 3), datetime.datetime(2023, 1, 1, 9, 0, 4)]


In [3]:
initial_price = 100.0
price_fluctuation_std = 0.1

# Simulate price data with sequential dependencies
prices = [initial_price]
for _ in range(1, num_ticks):
    # Simulate a small random change, ensuring sequential movement
    change = np.random.randn() * price_fluctuation_std
    new_price = prices[-1] + change
    # Ensure price doesn't go negative, though not strictly necessary for this simulation
    prices.append(max(0.01, new_price))

print(f"Generated {len(prices)} price data points.")
print(f"First 5 prices: {prices[:5]}")

Generated 1000 price data points.
First 5 prices: [100.0, 99.98783537052863, 99.99088507894005, 99.72678402783345, 99.93546857964915]


In [4]:
min_volume = 100
max_volume = 10000

# Simulate volume data
volumes = np.random.randint(min_volume, max_volume, size=num_ticks).tolist()

print(f"Generated {len(volumes)} volume data points.")
print(f"First 5 volumes: {volumes[:5]}")

Generated 1000 volume data points.
First 5 volumes: [8182, 6255, 9178, 820, 9005]


In [5]:
window_size = 10 # For a 10-tick moving average

# Calculate a simple moving average of prices as the market index
# Using pandas Series to easily apply rolling mean
market_index = pd.Series(prices).rolling(window=window_size).mean().tolist()

# For the initial ticks where rolling mean is not available, we can fill with the price itself or NaN
# For simplicity, let's fill initial NaNs with the initial price values
for i in range(window_size - 1):
    if pd.isna(market_index[i]):
        market_index[i] = prices[i]

print(f"Generated {len(market_index)} market index data points.")
print(f"First 5 market index values: {market_index[:5]}")

Generated 1000 market index data points.
First 5 market index values: [100.0, 99.98783537052863, 99.99088507894005, 99.72678402783345, 99.93546857964915]


In [6]:
market_data = pd.DataFrame({
    'timestamp': timestamps,
    'price': prices,
    'volume': volumes,
    'market_index': market_index
})

print("Combined data into a Pandas DataFrame.")

Combined data into a Pandas DataFrame.


In [7]:
print('First 5 rows of the generated market data:')
print(market_data.head())

print('\nSummary statistics of the generated market data:')
print(market_data.describe())

First 5 rows of the generated market data:
            timestamp       price  volume  market_index
0 2023-01-01 09:00:00  100.000000    8182    100.000000
1 2023-01-01 09:00:01   99.987835    6255     99.987835
2 2023-01-01 09:00:02   99.990885    9178     99.990885
3 2023-01-01 09:00:03   99.726784     820     99.726784
4 2023-01-01 09:00:04   99.935469    9005     99.935469

Summary statistics of the generated market data:
                           timestamp        price      volume  market_index
count                           1000  1000.000000  1000.00000   1000.000000
mean   2023-01-01 09:08:19.500000256    98.963652  5076.07700     98.974192
min              2023-01-01 09:00:00    95.883909   107.00000     95.978944
25%    2023-01-01 09:04:09.750000128    97.132906  2555.00000     97.131867
50%       2023-01-01 09:08:19.500000    99.608152  5133.00000     99.642687
75%    2023-01-01 09:12:29.249999872   100.400639  7515.25000    100.395969
max              2023-01-01 09:16:39   

Define Liquid Neural Network (LNN) architecture.


In [8]:
try:
    import ncps
    print("ncps library is already installed.")
except ImportError:
    print("ncps library not found. Installing now...")
    !pip install ncps
    import ncps
    print("ncps library installed successfully.")

ncps library is already installed.


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from ncps.wirings import AutoNCP
from ncps.torch import LTC

print("PyTorch modules and ncps.ltc imported successfully.")

PyTorch modules and ncps.ltc imported successfully.


In [10]:
class MarketLNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MarketLNN, self).__init__()
        self.rnn = LTC(input_size, hidden_size, wiring=AutoNCP(hidden_size, output_size=1))
        # The output of LTC is typically fed into a linear layer for specific output_size
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x should be of shape (batch_size, sequence_length, input_size)
        # LTC expects (batch_size, sequence_length, input_size)
        # Its output is (batch_size, sequence_length, hidden_size)
        out, _ = self.rnn(x)

        # We are interested in the prediction for the last time step
        # 'out' contains outputs for all time steps. We take the last one.
        last_output = out[:, -1, :]

        # Pass the last output through a linear layer to get the final prediction
        prediction = self.linear(last_output)
        return prediction

# Determine input and output sizes from the generated data
input_features = ['price', 'volume']
input_size = len(input_features)
output_size = 1 # Predicting market_index
hidden_size = 32 # A common choice for hidden units

print(f"Defined MarketLNN class with input_size={input_size}, hidden_size={hidden_size}, output_size={output_size}")

Defined MarketLNN class with input_size=2, hidden_size=32, output_size=1


In [11]:
print("Upgrading ncps library to ensure compatibility...")
!pip install --upgrade ncps
print("ncps library upgraded successfully.")

import torch
import torch.nn as nn
import torch.optim as optim
from ncps.wirings import AutoNCP
from ncps.torch import LTC

print("PyTorch modules and ncps.ltc imported successfully after upgrade.")

Upgrading ncps library to ensure compatibility...
ncps library upgraded successfully.
PyTorch modules and ncps.ltc imported successfully after upgrade.


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim
from ncps.wirings import AutoNCP # Keep for reference if needed elsewhere, but not directly used with LTC constructor here
from ncps.torch import LTC

class MarketLNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(MarketLNN, self).__init__()
        # In ncps version 1.0.1, LTC's constructor does not accept a 'wiring' argument.
        # It takes in_features and units directly.
        self.rnn = LTC(input_size, hidden_size) # LTC inherently has liquid properties

        # The output of LTC is typically fed into a linear layer for specific output_size
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # x should be of shape (batch_size, sequence_length, input_size)
        # LTC expects (batch_size, sequence_length, input_size)
        # Its output 'out' will be (batch_size, sequence_length, hidden_size)
        # 'hidden' will be the final hidden state (batch_size, hidden_size)
        out, _ = self.rnn(x)

        # We are interested in the prediction for the last time step
        # 'out' contains outputs for all time steps. We take the last one.
        last_output = out[:, -1, :]

        # Pass the last output through a linear layer to get the final prediction
        prediction = self.linear(last_output)
        return prediction

# Determine input and output sizes from the generated data
input_features = ['price', 'volume']
input_size = len(input_features)
output_size = 1 # Predicting market_index
hidden_size = 32 # A common choice for hidden units

print(f"Defined MarketLNN class with input_size={input_size}, hidden_size={hidden_size}, output_size={output_size} using LTC.")

# Instantiate an object of the MarketLNN model
model = MarketLNN(input_size, hidden_size, output_size)

print("\nInstantiated MarketLNN model:")
print(model)

Defined MarketLNN class with input_size=2, hidden_size=32, output_size=1 using LTC.

Instantiated MarketLNN model:
MarketLNN(
  (rnn): LTC(
    (rnn_cell): LTCCell(
      (make_positive_fn): Softplus(beta=1.0, threshold=20.0)
      (_clip): ReLU()
    )
  )
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


Set up data, training loop, and evaluation metrics.


In [13]:
from sklearn.preprocessing import StandardScaler

# 1. Extract features and target
features = market_data[input_features].values
target = market_data['market_index'].values.reshape(-1, 1) # Reshape for StandardScaler

print(f"Extracted features shape: {features.shape}")
print(f"Extracted target shape: {target.shape}")

Extracted features shape: (1000, 2)
Extracted target shape: (1000, 1)


In [14]:
feature_scaler = StandardScaler()
target_scaler = StandardScaler()

features_scaled = feature_scaler.fit_transform(features)
target_scaled = target_scaler.fit_transform(target)

print(f"Scaled features shape: {features_scaled.shape}")
print(f"Scaled target shape: {target_scaled.shape}")
print("Features and target scaled successfully.")

Scaled features shape: (1000, 2)
Scaled target shape: (1000, 1)
Features and target scaled successfully.


In [15]:
sequence_length = 20 # Number of previous ticks to consider for prediction

print(f"Sequence length defined as: {sequence_length}")

Sequence length defined as: 20


In [16]:
X_sequences = []
y_targets = []

for i in range(sequence_length, num_ticks):
    X_sequences.append(features_scaled[i-sequence_length:i])
    y_targets.append(target_scaled[i])

X_sequences = np.array(X_sequences)
y_targets = np.array(y_targets)

print(f"Created {len(X_sequences)} sequences.")
print(f"Shape of X_sequences: {X_sequences.shape}")
print(f"Shape of y_targets: {y_targets.shape}")

Created 980 sequences.
Shape of X_sequences: (980, 20, 2)
Shape of y_targets: (980, 1)


In [17]:
X_sequences_tensor = torch.tensor(X_sequences, dtype=torch.float32)
y_targets_tensor = torch.tensor(y_targets, dtype=torch.float32)

print(f"Shape of X_sequences_tensor: {X_sequences_tensor.shape}")
print(f"Shape of y_targets_tensor: {y_targets_tensor.shape}")
print("NumPy arrays converted to PyTorch tensors successfully.")

Shape of X_sequences_tensor: torch.Size([980, 20, 2])
Shape of y_targets_tensor: torch.Size([980, 1])
NumPy arrays converted to PyTorch tensors successfully.


In [18]:
from torch.utils.data import Dataset, DataLoader

class MarketDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

print("MarketDataset class defined successfully.")

MarketDataset class defined successfully.


In [19]:
full_dataset = MarketDataset(X_sequences_tensor, y_targets_tensor)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Full dataset size: {len(full_dataset)}")
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Batch size for DataLoaders: {batch_size}")
print("Train and validation DataLoaders created successfully.")

Full dataset size: 980
Training dataset size: 784
Validation dataset size: 196
Batch size for DataLoaders: 32
Train and validation DataLoaders created successfully.


In [20]:
loss_fn = nn.MSELoss()
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

print(f"Loss function initialized: {loss_fn}")
print(f"Optimizer initialized: {optimizer.__class__.__name__} with learning rate: {learning_rate}")

Loss function initialized: MSELoss()
Optimizer initialized: Adam with learning rate: 0.001


In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train() # Set model to training mode
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        predictions = model(batch_X)
        loss = loss_fn(predictions, batch_y)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

print(f"Device set to: {device}")
print("train_epoch function defined successfully.")

Device set to: cuda
train_epoch function defined successfully.


In [22]:
from torch.utils.data import Dataset, DataLoader

class MarketDataset(Dataset):
    def __init__(self, X, y, all_original_targets_scaled, sequence_length):
        self.X = X # X_sequences_tensor
        self.y = y # y_targets_tensor
        self.all_original_targets_scaled = all_original_targets_scaled # original target_scaled array
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # `self.y[idx]` is the current target, corresponding to original_target_scaled[idx_original_in_X_sequences + sequence_length]
        # X_sequences was created from `range(sequence_length, num_ticks)`
        # so `idx_in_original_target_scaled_for_current_y = idx + sequence_length`
        # The previous actual target needed for directional accuracy is target_scaled[idx_in_original_target_scaled_for_current_y - 1].
        original_target_index_for_current_y = idx + self.sequence_length
        previous_actual_target_scaled = self.all_original_targets_scaled[original_target_index_for_current_y - 1]
        return self.X[idx], self.y[idx], torch.tensor(previous_actual_target_scaled, dtype=torch.float32)

print("MarketDataset class redefined successfully to include previous target for directional accuracy calculation.")

full_dataset = MarketDataset(X_sequences_tensor, y_targets_tensor, target_scaled, sequence_length)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

batch_size = 32
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

print(f"Full dataset size: {len(full_dataset)}")
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Batch size for DataLoaders: {batch_size}")
print("Train and validation DataLoaders recreated successfully.")

MarketDataset class redefined successfully to include previous target for directional accuracy calculation.
Full dataset size: 980
Training dataset size: 784
Validation dataset size: 196
Batch size for DataLoaders: 32
Train and validation DataLoaders recreated successfully.


In [23]:
def evaluate_model(model, dataloader, loss_fn, device, target_scaler):
    model.eval() # Set model to evaluation mode
    total_loss = 0
    correct_directions = 0
    total_predictions = 0

    with torch.no_grad(): # Disable gradient calculation for evaluation
        for batch_X, batch_y, batch_prev_y in dataloader:
            batch_X, batch_y, batch_prev_y = batch_X.to(device), batch_y.to(device), batch_prev_y.to(device)

            # Forward pass
            predictions = model(batch_X)
            loss = loss_fn(predictions, batch_y)
            total_loss += loss.item()

            # Calculate directional accuracy
            # Inverse transform predictions and actuals to original scale for meaningful direction comparison
            # Note: target_scaler.inverse_transform expects a 2D array, so reshape batch_y and predictions
            # Also, batch_prev_y is already a tensor and might need to be flattened before inverse_transform if it's 1D
            # Ensure batch_prev_y is handled correctly if it's already a 1D tensor of single values

            # The batch_y and predictions are (batch_size, 1), which fits inverse_transform
            # batch_prev_y is (batch_size, 1) from the MarketDataset getitem

            actual_current = target_scaler.inverse_transform(batch_y.cpu().numpy())
            actual_prev = target_scaler.inverse_transform(batch_prev_y.cpu().numpy())
            predicted_current = target_scaler.inverse_transform(predictions.cpu().numpy())

            actual_change = actual_current - actual_prev
            predicted_change = predicted_current - actual_prev

            correct_directions += ((np.sign(actual_change) == np.sign(predicted_change)).sum())
            total_predictions += len(actual_change)

    avg_loss = total_loss / len(dataloader)
    directional_accuracy = correct_directions / total_predictions if total_predictions > 0 else 0
    return avg_loss, directional_accuracy

print("evaluate_model function defined successfully.")

evaluate_model function defined successfully.


Integrate Optuna for hyperparameter optimization.


In [24]:
import optuna

print("Optuna library imported successfully.")

Optuna library imported successfully.


In [25]:
def objective(trial):
    # 2a. Suggest hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    hidden_size = trial.suggest_int('hidden_size', 16, 128) # Expanded range for more exploration
    num_epochs = trial.suggest_int('epochs', 5, 30) # Expanded range for more exploration

    # 2b. Instantiate a new MarketLNN model, optimizer, and loss_fn
    # Ensure input_size and output_size are available (from global scope or passed)
    model = MarketLNN(input_size, hidden_size, output_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    # 2c. DataLoaders are already available in the global scope (train_dataloader, val_dataloader)

    # 2d. Implement a training loop
    for epoch in range(num_epochs):
        train_loss = train_epoch(model, train_dataloader, loss_fn, optimizer, device)
        # Optionally, print progress during optimization
        # print(f"Trial {trial.number}, Epoch {epoch+1}: Train Loss = {train_loss:.4f}")

    # 2e. Evaluate the model on the validation set and return the validation loss
    val_loss, directional_accuracy = evaluate_model(model, val_dataloader, loss_fn, device, target_scaler)

    # Optuna by default minimizes the objective. We want to minimize validation loss.
    # If we wanted to maximize directional accuracy, we'd return -directional_accuracy.
    trial.set_user_attr("directional_accuracy", directional_accuracy)

    print(f"Trial {trial.number}: LR={lr:.6f}, Hidden={hidden_size}, Epochs={num_epochs}, Val Loss={val_loss:.4f}, Dir Acc={directional_accuracy:.4f}")

    return val_loss

print("Optuna objective function defined successfully.")

Optuna objective function defined successfully.


In [26]:
study = optuna.create_study(direction='minimize')

print("Optuna study created with 'minimize' direction.")

[I 2026-01-14 23:34:20,822] A new study created in memory with name: no-name-4ce3a942-70ef-4a12-bf9d-3dc9744db77d


Optuna study created with 'minimize' direction.


In [27]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

def train_epoch(model, dataloader, loss_fn, optimizer, device):
    model.train() # Set model to training mode
    total_loss = 0
    for batch_X, batch_y, _ in dataloader: # Unpack 3 values, ignore the third with _
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        # Forward pass
        predictions = model(batch_X)
        loss = loss_fn(predictions, batch_y)

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(dataloader)

print(f"Device set to: {device}")
print("train_epoch function defined successfully.")

Device set to: cuda
train_epoch function defined successfully.


In [28]:
n_trials = 10 # Number of trials for Optuna to run
study.optimize(objective, n_trials=n_trials)

print("Optuna optimization finished.")
print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print(f"  Value: {trial.value:.4f}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")
print(f"  Directional Accuracy: {trial.user_attrs['directional_accuracy']:.4f}")

[I 2026-01-14 23:35:08,946] Trial 0 finished with value: 0.0974948491368975 and parameters: {'lr': 0.00040667072401347676, 'hidden_size': 115, 'epochs': 15}. Best is trial 0 with value: 0.0974948491368975.


Trial 0: LR=0.000407, Hidden=115, Epochs=15, Val Loss=0.0975, Dir Acc=0.5408


[I 2026-01-14 23:36:19,676] Trial 1 finished with value: 0.0019492390399266566 and parameters: {'lr': 0.03715789739285449, 'hidden_size': 95, 'epochs': 22}. Best is trial 1 with value: 0.0019492390399266566.


Trial 1: LR=0.037158, Hidden=95, Epochs=22, Val Loss=0.0019, Dir Acc=0.7602


[I 2026-01-14 23:37:08,249] Trial 2 finished with value: 0.008437828227345432 and parameters: {'lr': 0.0026240772391769194, 'hidden_size': 52, 'epochs': 16}. Best is trial 1 with value: 0.0019492390399266566.


...

Trial 7: LR=0.000179, Hidden=114, Epochs=12, Val Loss=0.9207, Dir Acc=0.5459


[I 2026-01-14 23:48:18,063] Trial 8 finished with value: 0.04745062227760043 and parameters: {'lr': 0.0008316955279815951, 'hidden_size': 34, 'epochs': 17}. Best is trial 1 with value: 0.0019492390399266566.


Trial 8: LR=0.000832, Hidden=34, Epochs=17, Val Loss=0.0475, Dir Acc=0.5969


[I 2026-01-14 23:50:54,166] Trial 9 finished with value: 0.002373271089579378 and parameters: {'lr': 0.04305744397443286, 'hidden_size': 106, 'epochs': 22}. Best is trial 1 with value: 0.0019492390399266566.


Trial 9: LR=0.043057, Hidden=106, Epochs=22, Val Loss=0.0024, Dir Acc=0.7245
Optuna optimization finished.
Number of finished trials:  10
Best trial:
  Value: 0.0019
  Params: 
    lr: 0.03715789739285449
    hidden_size: 95
    epochs: 22
  Directional Accuracy: 0.7602


Train final LNN model with optimized hyperparameters.


In [29]:
best_params = study.best_trial.params
best_lr = best_params['lr']
best_hidden_size = best_params['hidden_size']
best_epochs = best_params['epochs']

print(f"Retrieved best hyperparameters:\n  Learning Rate: {best_lr:.6f}\n  Hidden Size: {best_hidden_size}\n  Epochs: {best_epochs}")

Retrieved best hyperparameters:
  Learning Rate: 0.037158
  Hidden Size: 95
  Epochs: 22


In [30]:
final_model = MarketLNN(input_size, best_hidden_size, output_size)
final_model.to(device)

print(f"Final MarketLNN model instantiated with hidden_size={best_hidden_size}.")
print(final_model)

Final MarketLNN model instantiated with hidden_size=95.
MarketLNN(
  (rnn): LTC(
    (rnn_cell): LTCCell(
      (make_positive_fn): Softplus(beta=1.0, threshold=20.0)
      (_clip): ReLU()
    )
  )
  (linear): Linear(in_features=95, out_features=1, bias=True)
)


In [31]:
final_optimizer = optim.Adam(final_model.parameters(), lr=best_lr)
final_loss_fn = nn.MSELoss()

print(f"Final optimizer initialized: {final_optimizer.__class__.__name__} with learning rate: {best_lr:.6f}")
print(f"Final loss function initialized: {final_loss_fn}")

Final optimizer initialized: Adam with learning rate: 0.037158
Final loss function initialized: MSELoss()


In [32]:
print(f"Starting final model training for {best_epochs} epochs...")

for epoch in range(best_epochs):
    train_loss = train_epoch(final_model, train_dataloader, final_loss_fn, final_optimizer, device)
    print(f"Epoch {epoch+1}/{best_epochs}, Train Loss: {train_loss:.4f}")

print("Final model training complete.")

Starting final model training for 22 epochs...
Epoch 1/22, Train Loss: 0.8835
Epoch 2/22, Train Loss: 0.1375
Epoch 3/22, Train Loss: 0.0204
Epoch 4/22, Train Loss: 0.0090
Epoch 5/22, Train Loss: 0.0087
Epoch 6/22, Train Loss: 0.0077
Epoch 7/22, Train Loss: 0.0077
Epoch 8/22, Train Loss: 0.0075
Epoch 9/22, Train Loss: 0.0083
Epoch 10/22, Train Loss: 0.0065
Epoch 11/22, Train Loss: 0.0066
Epoch 12/22, Train Loss: 0.0045
Epoch 13/22, Train Loss: 0.0042
Epoch 14/22, Train Loss: 0.0037
Epoch 15/22, Train Loss: 0.0046
Epoch 16/22, Train Loss: 0.0038
Epoch 17/22, Train Loss: 0.0047
Epoch 18/22, Train Loss: 0.0030
Epoch 19/22, Train Loss: 0.0021
Epoch 20/22, Train Loss: 0.0023
Epoch 21/22, Train Loss: 0.0014
Epoch 22/22, Train Loss: 0.0014
Final model training complete.


In [33]:
val_loss, directional_accuracy = evaluate_model(final_model, val_dataloader, final_loss_fn, device, target_scaler)

print(f"\nFinal Model Evaluation on Validation Set:")
print(f"  Validation Loss (MSE): {val_loss:.4f}")
print(f"  Directional Accuracy: {directional_accuracy:.4f}")


Final Model Evaluation on Validation Set:
  Validation Loss (MSE): 0.0018
  Directional Accuracy: 0.7704


Simulate real-time tick-by-tick market index prediction.


In [34]:
current_sequence_buffer = list(features_scaled[num_ticks - sequence_length:])

print(f"Initialized sequence buffer with {len(current_sequence_buffer)} elements.")
print(f"First element of buffer: {current_sequence_buffer[0]}")
print(f"Last element of buffer: {current_sequence_buffer[-1]}")

Initialized sequence buffer with 20 elements.
First element of buffer: [-0.57538962  1.22368698]
Last element of buffer: [-0.70237084  0.64540655]


In [35]:
real_time_predictions = []
real_time_actuals = []

print("Empty lists `real_time_predictions` and `real_time_actuals` created.")

Empty lists `real_time_predictions` and `real_time_actuals` created.


In [36]:
simulation_ticks = 50 # Number of ticks to simulate in real-time
simulation_start_index = num_ticks - simulation_ticks

print(f"Starting real-time simulation for {simulation_ticks} ticks, from index {simulation_start_index} to {num_ticks-1}.")

# Loop setup. The actual loop will be in the next step to keep the code blocks concise.
# For now, just setting up the start and end of the simulation range.

Starting real-time simulation for 50 ticks, from index 950 to 999.


In [37]:
final_model.eval() # Ensure model is in evaluation mode

for i in range(simulation_start_index, num_ticks):
    # 4. For each incoming tick, extract its price and volume, and scale them
    current_tick_features = market_data[input_features].iloc[i].values.reshape(1, -1)
    scaled_current_tick_features = feature_scaler.transform(current_tick_features)

    # 5. Append the newly scaled features to the `current_sequence_buffer`
    # and remove the oldest entry to maintain a fixed `sequence_length`.
    current_sequence_buffer.append(scaled_current_tick_features[0])
    if len(current_sequence_buffer) > sequence_length:
        current_sequence_buffer.pop(0)

    # Ensure buffer always has the correct length before prediction
    if len(current_sequence_buffer) == sequence_length:
        # 6. Convert the `current_sequence_buffer` into a PyTorch tensor
        # reshape it to `(1, sequence_length, input_size)`, and move it to the `device`.
        input_sequence = torch.tensor(np.array(current_sequence_buffer), dtype=torch.float32).unsqueeze(0).to(device)

        # 7. Use the `final_model` to make a prediction on this tensor.
        with torch.no_grad(): # No gradients needed for prediction
            prediction_scaled = final_model(input_sequence)

        # 8. Inverse transform the predicted market index using `target_scaler`
        predicted_value = target_scaler.inverse_transform(prediction_scaled.cpu().numpy())
        real_time_predictions.append(predicted_value.item())

        # 9. Extract the actual market index for the current tick from `market_data`
        # and inverse transform it using `target_scaler`.
        actual_target_scaled = target_scaled[i]
        actual_value = target_scaler.inverse_transform(actual_target_scaled.reshape(1, -1))
        real_time_actuals.append(actual_value.item())
    else:
        # If buffer is not yet full, no prediction can be made for this tick yet.
        # This should only happen for the very first 'sequence_length' ticks
        # of the simulation range, but our simulation_start_index handles that.
        # We will append NaN or similar if we strictly need a value for every tick
        # but for this simulation, we'll only append when a prediction is made.
        pass

print(f"Completed real-time simulation for {simulation_ticks} ticks. Generated {len(real_time_predictions)} predictions.")

Completed real-time simulation for 50 ticks. Generated 50 predictions.


In [38]:
print('\nSimulated Real-time Predictions vs. Actuals (First 10 ticks):')
for i in range(min(10, len(real_time_predictions))):
    print(f"  Tick {i+1}: Predicted = {real_time_predictions[i]:.4f}, Actual = {real_time_actuals[i]:.4f}")


Simulated Real-time Predictions vs. Actuals (First 10 ticks):
  Tick 1: Predicted = 97.5118, Actual = 97.1765
  Tick 2: Predicted = 97.4465, Actual = 97.1752
  Tick 3: Predicted = 97.3467, Actual = 97.1725
  Tick 4: Predicted = 97.3136, Actual = 97.1611
  Tick 5: Predicted = 97.2569, Actual = 97.1388
  Tick 6: Predicted = 97.2098, Actual = 97.1295
  Tick 7: Predicted = 97.1979, Actual = 97.1246
  Tick 8: Predicted = 97.1882, Actual = 97.1207
  Tick 9: Predicted = 97.1960, Actual = 97.1097
  Tick 10: Predicted = 97.1957, Actual = 97.1232


Brief summary: simulate synthetic tick-by-tick market data, train a Liquid Neural Network with Optuna-tuned hyperparameters to predict the market index, and run a simple real-time-style prediction loop on the trained model.